# NB_02_risk_field_validation - Risk Field Empirical Validation

**Purpose.** Drive the RiskOracle against a live CARLA world to produce top-down risk-grid renderings - the qualitative evidence behind the calibration choices in section 4.2 of the thesis.

**Inputs.** a CARLA server with a loaded map, the constants in MIREIA/core/constants.py.

**Outputs.** output/risk_video.mp4 (top-down DRF heatmap over time), per-tick scalar risk values.

**How to run.** Run cells top-to-bottom. The notebook spawns traffic, builds the SimulationBridge, bakes the static-risk layer, and renders a 300-frame risk video.

**Position in the workflow.** Sits alongside sections 4.2-4.3 of the thesis. Independent of the training pipeline.


## 1. Launch CARLA

Spawn the CARLA server as a background process. Skip this cell if CARLA is already running.


In [ ]:
# Run the file CarlaUE4.exe in a background process
import os
import subprocess

command = "CarlaUE4.exe"
# command = "CarlaUE4.exe --quality-level=Low"
#-RenderOffScreen

subprocess.Popen(command, shell=True)

#subprocess.Popen(["CarlaUE4.exe"])

## 2. Connect, Enable Synchronous Mode

Open a client connection and switch the world into synchronous mode at the fixed time step used throughout MIREIA (`fixed_delta_seconds = 0.05`, i.e. 20 Hz).


In [ ]:
import carla
import random

# Connect to the client and retrieve the world object
client = carla.Client('localhost', 2000)
client.set_timeout(20.0)
world = client.get_world()
world_map = world.get_map()

# Enable synchronous mode so the simulation only advances when we call world.tick()
settings = world.get_settings()
settings.synchronous_mode = True
settings.fixed_delta_seconds = 0.05
world.apply_settings(settings)

## 3. Background Tick Thread

In synchronous mode the simulation only advances when `world.tick()` is called. A background thread ticks the world at 20 Hz so the rest of the notebook can interact with a live scene without manually ticking.


In [ ]:
# Use a background thread to tick the world every 0.05 seconds
import threading
import time

def tick_world():
    previous_time = time.time()
    while True:
        current_time = time.time()
        if current_time - previous_time >= 0.05:
            world.tick()
            previous_time = current_time
        else:
            time.sleep(0.01)

tick_thread = threading.Thread(target=tick_world, daemon=True)
tick_thread.start()

## 4. Spawn Ego and Traffic

Use `TrafficHandler` to spawn the ego vehicle, surrounding traffic, and pedestrians with a fixed seed so the scene is reproducible.


In [ ]:
from MIREIA.simulation.traffic_handler import TrafficHandler

spawner = TrafficHandler(client, world, seed=42)
ego_vehicle = spawner.spawn_ego(autopilot=True)
spawner.spawn_vehicles(n=75, safe=True)
spawner.spawn_pedestrians(n=50, pct_running=0.3, pct_crossing=0.25)

## 5. Bridge - Per-Tick State Snapshot

The `SimulationBridge` collapses everything the risk oracle needs (ego kinematics, dynamic and static obstacles, environment state, waypoints) into a single immutable snapshot per tick.


In [ ]:
from MIREIA.simulation.bridge import SimulationBridge
bridge = SimulationBridge(world)

print(bridge.ego)
print(bridge.dynamic_obstacles)
print(bridge.env_state)
print(bridge)
bridge.update()
print(bridge.waypoints.get_closest_waypoint(bridge.ego.x, bridge.ego.y))

print(len(bridge.get_static_obstacles()))

## 6. Sensors and Frame Capture

Attach an ego-mounted RGB camera and a top-down map view. The map view is what the risk-grid renderer will overlay the heatmap onto.


In [ ]:
from MIREIA.simulation.sensors import SensorManager
sensors = SensorManager(world, world_map, ego_vehicle, save_dir="output", map_resolution=(3000, 3000))
#sensors.clean_output_directory()

In [ ]:
sensors.save_ego_frame()
sensors.save_map_frame()


## 7. Risk Oracle and Static-Risk Bake

Instantiate the `RiskOracle` and pre-bake the static-risk layer (road boundaries, traffic lights, parked vehicles, crosswalks) onto a uniform grid covering the map. Per-frame evaluation then samples this baked grid by bilinear interpolation rather than re-evaluating the static-layer equation point-by-point.


In [ ]:
from MIREIA.core.physics import RiskOracle
from MIREIA.analysis.plotter import draw_risk_heatmap_3d
from MIREIA.analysis.plotter import Grid
from MIREIA.analysis.visualizer import RiskGridVisualizer

oracle = RiskOracle()
map_grid = Grid(center_x=0, center_y=0, resolution=0.5, size = 500)
baked_static_risk = oracle.bake_static_risk(map_grid, bridge)


## 8. Per-Tick Scalar Risk + Risk Grid

Evaluate the full DRF on a 100 m grid centred on the ego at the current tick, then expose it to `RiskGridVisualizer` for top-down rendering.


In [ ]:
bridge.update()
ego_x, ego_y = bridge.get_ego_kinematics().x, bridge.get_ego_kinematics().y


grid = Grid(center_x=ego_x, center_y=ego_y, size=100.0, resolution=0.25)
risk_grid = oracle.calculate_risk_map(grid, bridge, baked_static_risk=baked_static_risk)

risk_grid = oracle.calculate_risk_map(grid, bridge, baked_static_risk=baked_static_risk)
viz = RiskGridVisualizer(risk_grid, world, bridge, vmax=6.0)
viz.render(save_path="output/risk_heatmap.png")  # or just viz.render() to display inline

## 9. Single-Point Scalar Query Loop

Advance the world a few ticks and query `calculate_scene_risk` at the ego location. This is the same call path that the dataset logger uses to produce the ground-truth label for each recorded frame.


In [ ]:
import time
scene_or_map = True
for i in range(1):
    # Calculate the risk at the exact point of the ego vehicle
    world.tick() # Advance the simulation by one tick (0.05 seconds)
    bridge.update()  # Only update dynamic obstacles and ego state
    ego_kinematics = bridge.get_ego_kinematics()
    if scene_or_map:
        risk = oracle.calculate_scene_risk((ego_kinematics.x, ego_kinematics.y), bridge, baked_static_risk=baked_static_risk)
        print(f"Scene Risk at Ego Position: {risk:.4f}")
    else:
        grid = Grid(center_x=ego_kinematics.x, center_y=ego_kinematics.y, size=100.0, resolution=0.25)
        risk_grid = oracle.calculate_risk_map(grid, bridge, baked_static_risk=baked_static_risk)
        print(f"Risk Map: min={risk_grid.lowest_risk:.4f}, max={risk_grid.highest_risk:.4f}")
    #time.sleep(0.1)  # Simulate time delay between updates

## 10. Route Construction (optional)

Construct a CARLA route from the bridge waypoints - used here only as a sanity check that the waypoint graph is consistent with what the oracle sees.


In [ ]:
from MIREIA.simulation.routes import create_route_from_waypoints

waypoints = bridge.waypoints
route = create_route_from_waypoints(waypoints)

## 11. Render Top-Down Risk Video

Drive the visualiser forward for 300 frames and render the top-down DRF heatmap as a video. The figures exported here are the qualitative evidence for section 4.3 of the thesis.


In [ ]:
viz.render_video("output/risk_video.mp4", n_frames=300, grid_size=200.0, grid_resolution=1.0, baked_static_risk=baked_static_risk)

## 12. Standalone Risk-Field Explorer (no CARLA)

`DummyRiskVisualizer` is a self-contained interactive widget over the same `RiskOracle`. It synthesises an ego, an environment and a small set of scripted obstacles (Following, Cut-In, Oncoming, Traffic Jam), then renders the field with sliders so the effect of speed, friction and visibility on the heatmap can be inspected without a running simulator.

The widget needs an interactive backend (Jupyter's default `inline` only produces static images, so the sliders and radio buttons would not respond). The cell below selects the `tk` backend via the `%matplotlib tk` magic, which opens the visualiser in a separate window and uses only the standard-library `tkinter` (no extra dependency). Alternatives if `tk` is unavailable: `%matplotlib qt` (requires `PyQt5`/`PySide`) or `%matplotlib widget` (requires `ipympl`).


In [ ]:
%matplotlib tk
from MIREIA.analysis.visualizer import DummyRiskVisualizer

visualizer = DummyRiskVisualizer()
